<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="https://www.uoc.edu/portal/_resources/common/imatges/sala_de_premsa/noticies/2016/202-nova-marca-uoc.jpg", align="left" width="380" height="120">

</div>
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">M2.893 · Anàlisi de textos</p>
<p style="margin: 0; text-align:right;">Màster Universitari de Ciència de Dades</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Estudis d'Informàtica, Multimèdia i Telecomunicacions</p>
</div>
</div>
<div style="width: 100%; clear: both;">
<div style="width:100%;">&nbsp;</div>

# PRA 2: Deep Learning per a l'anàlisi de textos

En aquesta pràctica revisarem i aplicarem els coneixements apresos en el últims mòduls del curs. Treballarem els següents temes:

1. **Traducció automàtica(TA)**: amb 'custom embeddings' i amb 'embeddings preentrenats'.
2. **NER i NEL**: Entrenament de models de detecció d'entitats anomenades (NER), detecció i classificació.  Detección de entitats anomenades basant-nos en Wikidata aplicada a NER.

També inclourem altres temes transversals treballats al llarg de l'assignatura.

#0 Connexió amb 'Google Drive'

Aquesta secció realitza la connexió amb `drive` i estableix el directori arrel en el que s'emmagatzemem tots els recursos necessaris per a executar el notebook.

El 'path' de treball s'emmagatzema a la variable `my_path_pra2`.

**Estructura de directoris**

S'estableix el directori arrel segons la variable `my_path_pra2`. En aquest directori s'emmagatzemaran els arxius i directoris necessaris per a l'execució del notebook. L'estructura i continguts són els següents:

    * directori `TA` on s'emmagatzemen les dades i recursos per a realitzar la traducció automàtica; conté:
      * glove.42B.300d.txt    # carregat per l'usuari.
      * nld.txt      # carregat per l'usuari.
      * directori `model` on s'emmagatzemen els *best model* de l'entrenament dels models de traducció automàtica:
        * model_ta_en_de-g.keras    # 'best model' generat per l'entrenament de TA amb 'embeddings' preentrenats
        * model_ta_en_de.keras      # 'best model' generat por l'entrenament de TA amb 'custom' embeddings.
    * directori `NER` amb els arxius necessaris per a la pràctica NER:
        * Directori `output_ner`   on s'hi trobaran els *model-best* y *model-last* entrenats per aquest notebook.
        * config.cfg    # carregat per l'usuari.
        * test.txt      # carregat per l'usuari.
        * test.spacy    # Conversió de test.txt al format spacy
        * train.txt     # carregat per l'usuari.
        * train.spacy   # Conversió de train.txt al format spacy
        * valid.txt     # carregat per l'usuari.
        * valid.spacy   # Conversió de valid.txt al format spacy



**Execució d'aquest notebook en un entorno no `Colab`.**

Si no s'executa aquest notebook a Google Colab, substituir aquesta secció (*0. Connexió amb Google Drive*) per la corresponent a la configuració desitjada, tenint en compte que cal disposar d'una GPU amb, almenys, 15 GB de memòria RAM.

**Execució d'aquest notebook en un entorno `Colab`.**

Si s'executa aquest notebook a Colab, cal utilitzar al menys una GPU del tipus 'T4 GPU' o superior. Tenir en compte que si s'utilitza el servei gratuït de Colab, aquestes GPU no estan disponibles permanentment i, quan ho estan, només se'n pot disposar mentre duren les 'compute units' assignades a l'usuari o per límits de disponibilitat de GPUs de Google. Quan no hi ha disponibilitat, cal esperar a una nova assignació. Google no publica el mètode d'assignació o els [terminis de disposició](https://research.google.com/colaboratory/faq.html#usage-limits) de GPUs. Consulta a Colab (Opció de menú 'Runtime' >>>> 'View resources') la disponibilitat en cada moment de Compute Units i GPUs.

En tots els casos, aquest notebook pressuposa l'estructura de directoris de treball descrita.

In [ ]:
# Accedir a Colab myDrive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Update path as needed; maintain the structure described.
my_path_pra2 = "/content/drive/MyDrive/UOC/20242_ADT/PRA2"

if os.path.exists(my_path_pra2):
    try:
        os.chdir(my_path_pra2)
        print(f"Root directory: '{os.getcwd()}'")
    except Exception as e:
        print(f"Error changing directori¡y: '{my_path_pra2}'. Error: {e}")
else:
    print(f"Directory'{my_path_pra2}' doesn't exist")

Directorio raíz cambiado a: '/content/drive/MyDrive/UOC/20242_ADT/PRA2'


# Imports

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################

import keras
from keras import optimizers
from keras.utils import pad_sequences
from keras import layers
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.models import Sequential, load_model
from keras.layers import Dense, LSTM, Embedding, RepeatVector

# ... Other imports ...

# 1 Traducció Automàtica (TA) (7 puntos)




En aquesta primera part de la pràctica es demana resoldre els exercicis usant la llibreria **KERAS**.

## 1.1 TA amb Custom Embeddings

L'objectiu d'aquest apartat és entrenar un model de traducció automàtica entre dos idiomes escollits a partir de l'arxiu escollit, seguint els mateixos passos que el notebook de *Machine Translation* i l'exemple proporcionat per al desenvolupament d'aquesta pràctica `Exemple_PRA2`.

**Implementació:** Seguint els passos treballats en el notebook de traducció automàtica, implementar i entrenar un model de traducció automàtica, de l'**idioma origen** a l'**idioma destí**.
* En aquest notebook d'exemple s'utilitza com a dimensión de la capa d'embedding (*embedding_vec_length*) el valor de 200. Més endavant, es variarà aquesta dimensió i es revisaran els resultats.
* La longitud de seqüència es determinant en la duració del temps de procés i en el consumo de memoria del processador durant l'entrenament. A Google Colab es pot cancel·lar el programa si es consumeix tota la memòria disponible. Provar amb diferents longituds, des de 4 (molt reduïda i, per tant, produirà una qualitat baixa de traducció), fins a longituds més grans (8, 12, 16).

* Finalment, es mostrarà com aplicar el model entrenat amb exemples de l'arxiu de dataset.


### 1.1.0 Hiperparàmetres

Notar que:

* En aquest exercici utilitzarem 'max_text_length' tant per a la seqüència d'entrada com de sortida.

In [ ]:
# Model hyperparameters

max_text_length =           # Maximum number of tokens allowed per input  and output sequence
embedding_vec_length =      # Dimensionality of the dense vector representing each token.
units =                     # LSTM Layer Units
epochs =                    # Training epochs
patience =                  # early stopping patience
batch_size =                # Number of training examples per step

### 1.1.1 Preparació de dades (1 punt)

* Primer prepararem les dades seleccionades (tingueu en compte que l'*idioma origen* ha de ser **anglès**), per a que es puguin llegir correctament i tinguin el format adequat per a la pràctica.

**a. Carreguem les dades des de la font seleccionada.**

*Sortides esperades:*
- Longitud del dataset.
- Almenys 3 files de dades que mostrin els textos de l'idioma origen i la respectiva traducció.

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################


**b. Preprocessar les dades, per a eliminar la puntuació i convertir a minúscula.**

*Sortida esperada:* Hauràs de mostrar el conjunt de dades normalitzat. Per exemple, la frase en l'idioma origen, "Hello, world!" es transformarà en l'idioma destí "Hola, món".en "hello world".

In [ ]:
#############################################
# SOLUCIÓ                                  #
#############################################




**c. Per a tenir una idea de la mida dels textos a analitzar, en funció de la quantitat de paraules, visualitzem les dades resultants amb un histograma.**

*Sortida esperada:* Dos histogramas que mostrin la quantitat de tokens dels textos del corpus, un per als vectors de l'idioma origen i l'altre per als de destí.

In [ ]:
#############################################
# SOLUCIÓ                                  #
#############################################

# Plot Source language



In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################

# Plot Target language

**d. Calculem el vocabulari tant el de l'idioma origen, com el de l'idioma destí, i imprimim el nombre de paraules**.

*Resultat esperat:*  
* Es visualitzaran dos nombres, el de paraules del vocabulari de l'idioma origen i el de destí, després d'haver aplicat el preprocessament i la tokenització.
* Llista dels 10 primers tokens de cada idioma.

In [ ]:
#############################################
# SOLUCIÓ                                  #
#############################################


**e. Separem els conjunts d'entrenament per idioma i els codifiquem.**

En aquest pas, se separen les dades en dos conjunts: un per a entrenament (*train*) i l'altre per a prova (*test*), utilitzant una divisió del 80% per a entrenament i 20% per a prova.

*Sortida esperada:* tres primeres files del dataset d'entrenament *train*.


In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################



**f. Definir i aplicar una funció per a codificar les seqüències**

En aquest pas, els dos conjunts de dades creats en el punt anterior,  seran codificats usant **tokenització** i **padding** per a assegurar que totes les seqüències d'un mateix idioma tinguin la mateixa longitud.

**Important:** Per a dur a terme un primer experiment, *depenent de la capacitat de processament disponible de cadascú*, se suggereix ajustar el valor del paràmetre **longitud de seqüència*, *fins a trobar el valor més alt possible que permeti entrenar el model encoder-decoder en un temps acceptable.*

El paràmetre **longitud de seqüència** té un impacte important en l'entrenament del model. Un valor alt permet al model capturar més context de les frases, la qual cosa és crucial per a traduir correctament oracions complexes; no obstant això, si la longitud és massa curta, el model pot truncar frases importants, perdent informació clau.

Ús de memòria y eficiència computacional:

Majors longituds requereixen més memòria, ja que el model defineix matrius més grans per a representar les seqüències. Mentre que longituds curtes són més eficients en termes de recursos, però poden sacrificar precisió si les oracions reals excedeixen aquest límit amb freqüència.

Finalment, en traducció automàtica, les longituds de les seqüències en l'idioma origen i destí no sempre han de ser iguals (per exemple, una oració en anglès pot ser més curta que la seva equivalent en alemany).

Considerant tot l'anterior, si es disposa d'infraestructura amb GPU, se suggereix iniciar amb un valor máxim de 12 (o proper) i mínim de 4.

Si durant l'entrenament, es produeixen problemes (per limitació d'infraestructura), es podrà tornar a aquest pas per a fixar el valor mínim de 4 per a la longitud de seqüència de tots dos idiomes (tot i que els resultats de la traducció no seran de qualitat)

**Sortida esperada:** Mida de cada arxiu i mostra de les tres primeres seqüències codificades de l'arxiu d'entrenament.

In [ ]:
#############################################
# SOLUCIÓ                                  #
#############################################



### 1.1.2 Definició del model encoder-decoder i entrenament (2 punts)

**a. Definim el model *encoder-decoder* basant-nos en el notebook vist a l'assignatura**, i l'instanciem amb una capa d'embedding per a les frases de la **llengua origen** i la dimensió de la última capa com la del vocabulario de la **llengua destí**.

**Important:** Per a la definició del model, considerar els següents paràmetres i valors referencials:

* Com a quantitat de **units** treballar, inicialment, amb el valor de 512. El nombre d'unitats o cel·les de memòria de cada capa LSTM defineix la dimensionalitat de l'espai intern en el que la LSTM processa i representa la informació al llarg del temps; és a dir, és la mida del vector de l'estat ocult *hidden state* i de l'estat de cel·la *cell state* que la LSTN manté per a capturar patrons i dependències de les seqüències d'entrada.

* A major nombre de *units*, augmenta la capacitat del model per a modelar relacions complexes i dependències a llarg termini en el text, la qual cosa és clau per a la traducció automàtica, on el context pot comprendre vàries paraules o frases. No obstant això, un valor alt incrementa el nombre de paràmetres i, per tant, requerirà més memòria i temps de càlcul; a més, creix el risc de sobreajustament si les dades d'entrenament no són suficients.



* Longitud dels vectors d'embeddings *embedding_vec_length*: establir-lo en 200; aquest és un valor referencial que podría ser ajustat segons la mida del vocabulari, la complexitat de l'idioma i els recursos disponibles. Més endavant, *exercici 1.1.3* es demanar variar aquest valor.

**Resultat esperat:** s'haurà instanciat un modelo encoder-decoder. Aquest model està dissenyat per a processar i traduir textos de l'**idioma origen** a l'**idioma destí** utilitzant capes d'embedding i LSTM.

*Sortida esperada*: Utilitzar el mètode *mt_model.summary()* per a visualitzar l'estructura i configuració del model, incloent el nombre de paràmetres i la disposició de les capes.

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################



**b. Compilem el model**

**Resultat esperat:** el model compilat i llest per a ser entrenat. S'utilitzarà l'optimitzador *RMSprop* amb una tasa d'aprenentatge de *0.001* i la funció de pèrdua *sparse_categorical_crossentropy*.


*Sortida esperada*: Utilitzar el mètode mt_model.summary() per a visualitzar l'estructura i configuració del model, incloent el nombre de paràmetres i la disposició de les capes.

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################




**c. Entrenem i guardem el model.**

**Important:** El model pot trigar hores si es disposa de CPU, molt menys amb GPU. Colab permet l'ús de GPU en general, si no se'n fa un ús intensiu i continuat. Per tant, habilitar només la GPU quan es necessiti per a entrenar i predir. Deshabilitar-la (i per tant reiniciar l'entorn) per a executar les cel·les no necessàries per a entrenar.

* Per tant, per a provar el funcionament, recomanem llençar l'entrenament **només amb una època** i comprovar-ne el funcionament. Un cop tenim clar que el sistema funciona, incrementar el valor (per exemple a 50 o 100, depenent de com evoluciona el model amb cada *epoch*)

* Si durant l'entrenament, Colab no pot carregar el model en memòria, recomanem disminuir el valor de **longitud de paraula** a 4 i el nombre de **units** a 128, d'aquesta manera es podrà completar el procés, tot i que, probablement, els resultats no seran bons.

* Revisar el *Notebook d'Exemple*, en el que es proporcionen pautes i guies per a dur un millor control de les execucions quan s'han de realitzar reinicis de sessió o s'exhaureix la memòria.

*Sortida Esperada*

* Path de la carpeta on s'ubicarà el best model
* Gràfica validation_loss / training_loss

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################





### 1.1.3 Predir

**a. Generar prediccions.**

Un cop entrenat el model, aplicar el mètode *predict()* a l'arxiu de test per a obtenir les prediccions.

**Important:** Si degut a les limitacions d'infraestructura, no es pot realitzar la predicció amb la totalitat de l'arxiu de testX, utilitzar-ne un subconjunt.

*Sortides Esperades*:
* shape de les raw predictions (raw_preds) i
* shape predictions (preds)
* print de les dues primeres prediccions

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################




**b. Visualització de resultats.**

Visualitzem els resultats de les prediccions amb els valors esperats.

**Resultat esperat:** prediccions traduïdes de les primeres 10 entrades del conjunt de prova. Aquestes prediccions es mostraran amb els textos esperats a efectes de comprovar la bondat de la predicció.

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################



f. **Pregunta d'anàlisi:** Depenent dels resultats obtinguts en la predicció (valors reals vs. valors generats), per què creus que no són bons i com creus que poden obtenir-se millors resultats?

**Resposta a la pregunta**

-







### 1.1.4 Experimentació amb diferentss resultats (1,5 puntos)

En aquest apartat podríem analitzar com afecta a la qualitat de la traducció la variació de diferents paràmetres del model com ara:
* longitud d'embeddings (*embedding_vec_length*),
* longitud de seqüència (*max_text_length*),
* número de units (*units*),
* batch size,
* epochs,
* ...

No obstant això, degut a que no sempre ens trobarem amb GPUs lliures, aquí ens limitarem a experimentar amb els paràmetres:
* *embedding_vec_length* i
* *max_text_length*.

**Important:** durant les execucions, depenent del model i del consum de memòria actual, la predicció pot cancel·lar si s'exhaureix la memòria disponible. En cas de cancel·lació, recarregar el model des de local (veue apartat 1.1.2.2 del `Notebook d'exemple`) i predir només un subconjunt de l'arxiu de test (tot i que en aquest cas no seran vàlides les magnituds de medició de qualitat del sistema)  

A més, suggerim que després de cada entrenament es realitzi una còpia del model entrenat i s'emmagatzemi en local (*'model/...'*) per a, en cas de cancel·lació, no haver de realitzar de nou l'entrenament, associant al nom de la còpia els paràmetres amb els que ha estat entrenat.


**a. Experimentar amb el valor de longitud de embedding** (*embedding_vec_length*)

Analitzar com un increment en la mida dels vectors de embedding afecta al rendiment d'un model de traducció automàtica de **l'idioma origen** a **l'idioma destí**.

**Resultat esperat:** S'imprimirà el resultat que mostri el rendiment del model creat per a diferents mides d'embeddings (inicialment s'ha treballat amb 200, també es podria experimentar amb valors com ara 50 i 300). Cada resultat constarà de la mida de l'embedding seguit d'un **score** que n'indiqui l'efectivitat del model calculat amb *model.evaluate()*.



In [ ]:
#############################################
# SOLUCIÓN                                  #
#############################################



**b. Exercici opcional: Experimentar amb el valor de longitud de seqüència** (*max_text_length*)

Analitzar com un increment/reducció de la longitud de seqüència impacta en la qualitat del model de traducció.

**Resultat esperat:** S'imprimirà el resultat que mostri el rendiment del model per a una longitud de seqüència superior o inferior a l'establert (per exemple, si es va inicialitzar el model preliminar amb el valor de 8, aquí es podria provar amb 4 i 12). El resultat constarà del valor de la longitud, i del score que indica l'efectivitat del model calculat amb *model.evaluate()*

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################


Segons els resultats obtinguts en aquest exercici 1.1.3, discutir en el *document d'Anàlisi* les diferències trobades.

## 1.2 TA amb Embeddings preentrenats (2,5 punts)


En aquest apartat repetirem l'exercici anterior carregant a la capa d'embedding els pesos d'un model `GloVe` entrenat per a l'anglès.

Aquest apartat 1.2 pot executar-se en diferents sessions de treball; no depèn de les seccions anteriors a excepció de:
* Executar l'apartat *0. Connexió amb Drive* (o les cel·les que s'hagin definit per a altres entorns no Colab).
* Executar l'apartat *Imports*
* Executar l'apartat *1.1.0 Hiperparàmetres*. **IMPORTANT** El paràmetre *embedding_vec_length* ha de coincidir amb la dimensió del vector de GloVe (inicialment 300)
* Executar la cel·la de preparació de dades (secció 1.1.1)





### 1.2.1 Càrrega de `GloVe`

**a. Començarem carregant el model `Glove`per a l'anglès.**

Podeu utilitzar [`'glove.42B.300d.txt'`](https://www.kaggle.com/datasets/yutanakamura/glove42b300dtxt).

**Sortida esperada:** mida de l'objecte carregat, utilitzar *len()*.

In [ ]:

embeddings_index = {}

glove      ="glove.42B.300d.txt"
glove_path = os.path.join(f"{my_path_pra2}/TA/", glove)    #'my_path_pra2' definida en sección 0


print(f"Attempting to open glove file: {glove_path}")
try:
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
        print(f"Successfully opened glove file: {glove}. Embeddings index dictionary created. Key: word. Value: Embedding")
except Exception as e:
    print(f"ERROR: File not found at opening file: {e}")

print(f"Number of words: {len(embeddings_index)}")

i = 0
embedding_vec_length_glove = None
for word, emb in embeddings_index.items():
    embedding_vec_length_glove = len(emb)
    print(f"word: '{word}', first embedding values: {emb[:20]}... ")
    i += 1
    if i > 9:
        break

print(f"Embedding vector length GloVe: {embedding_vec_length_glove}")


### 1.2.2 Definició del model

**a. Construir la matriu d'embeddings.**

A continuació, hem de construir la matriu d'embeddings.

Per a no carregar tot el vocabulari del model, filtrarem només aquelles entrades presents en el vocabulari del tokenitzador que utilitzarem.

A més, inclourem a la matriu de vectors els índexs de les entrades, paraules, que no constin en el model `glove` carregat. Aquests vectors se solen inicialitzar amb zeros o amb el resultat d'una distribució N (0, 1).

**Sortida esperada:**:
* Dimensió de la matriu d'embeddings
* Nombre de paraules de l'idioma origen que consten a Glove i les que no hi consten
* 10 Primeres paraules que consten a Glove y 10 primeres que no hi consten.
* Imprimir els 3 primers elements de la matriu d'embeddings.

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################



**b. Inicialitzar la capa d'embeddings.**

Per a inicialitzar una capa d'embeddings amb pesos predefinits s'utilitza l'argument `weights`.


A més, valorar si el paràmetre `trainable`es considera 'True' o 'False' en funció de si existeixen paraules del vocabulari de l'idioma origen que no existeixen a GloVe. Realitzar proves amb tos dos valors.


*Sortida Esperada*:

Dimensions de la capa de Embedding

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################



**c. Definició del nou model considerant els pesos del model preentrenat.**

 Implementa i entrena de nou un model de traducció automàtica de **l'idioma origen** a  **l'idioma destí**; aquest cop carregant els pesos de la capa embedding a partir del model `Glove` preentrenat en anglès i disponible a *glove.42B.300d.txt*.

 *Solució esperada*:
 'summary' de la definició del model

In [ ]:
#############################################
# SOLUCIÓ                                  #
#############################################




**d. Compilem el model**

 *Solució esperada*:
 'summary' de la definició del model

In [ ]:
#############################################
# SOLUCIÓ                                 #
#############################################



### 1.2.3 Entrenament del model

**Entrenar i guardar el model.**

Tot i que aquest entrenament és més 'lleuger' que l'anterior, recomanem l'ús de GPU si és viable.

**Suggeriments:**

- Provar amb diferents valors *batch_size*.

- Observar com evoluciona el model després de cada *epoch*; en caso de no observar millores, es pot disminuir el seu valor.

- Revisar el `Notebook d'Exemple`, secció 1.2.3.2, en el que proporcionem pautes i guies per a dur un millor control de les execucions quan s'ha de reiniciar la sessió o s'exhaureix la memòria.

*Sortida Esperada*

* Path de la carpeta on s'ubicarà el best model
* Plot validation_loss / training_loss

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################


### 1.2.4 Generar prediccions

En aquest pas, aplicar el model per a generar les prediccions utilitzant l'arxiu de test.


*Sortides Esperades*:
* shape de les raw predictions (raw_preds) i
* shape predictions (preds)
* print de les dues primeres prediccions
* Prediccions traduïdes de les primeres 10 entrades del conjunt de prova. Aquestes prediccions es mostraran: text idioma origen, text idioma destí real, text predit.

**Suggeriment:** Si durant l'execució, cancel·la la predicció per exhaurir-se la memòria disponible, recarregar el model des de local (veure apartat 1.2.3.2 del notebook exemple) i predir només per a un subconjunt de l'arxiu de test (tot i que en aquest cas no seran vàlides les magnituds de medició de qualitat del sistema).   

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################


# 2 Detecció de NER i NEL (3 punts)



En aquesta segona part, ens enfocarem en la detección de entidades anomenades (`NER`).

A més, experimentarem amb 'Named Entity Linking' (`NEL`) per a buscar entitats enllaçades a una base de coneixement (KB), en aquest cas *Wikidata*. Identificarem els enllaços a Wikidata de certes entitats d'un text utilitzant la API de Wikidata.

**Aquest apartat pot executar-se aïlladament**, no depèn de l'apartat anterior, a excepció d'executar l'apartat *0. Connexió amb Drive* (o les cel·les que s'hagin definit per a altres entorns no Colab).

## 2.1 Detecció de NER (2 punts)

En aquesta primera subsecció, detectarem entitats anomenades (`NER`), utilitzant tant spaCy com transformers.


### 2.1.1 Detecció d'entitats anomenades (NER) utilitzant spaCy.

Per a detectar NER utilitzarem el model `en_core_web_sm` de spaCy.

**a. Instal·lar llibreries i model de llenguatge a utilitzar.**

In [ ]:
!pip install spacy numpy

In [ ]:
!python -m spacy download en_core_web_sm

**`Important:`**

Si la descàrrega de *'en_core_web_sm'* es realitza des de Google Colab y, després de la descàrrega, es mostra el missatge 'Restart to reload dependencies', executar l'opció de menú 'Runtime>>>Restart session'. A continuació tornar a executar la secció 0 d'aquest notebook i seguir amb el següent apartat 2.1.1.b.

**b. Definir funcions per a imprimir els resultats de la detecció.**

In [ ]:
def get_tokens_to_print(model, text):
  """Print tokens of the text and its relevant attributes.

    Parameters:
      model (spaCy model): spaCy model used for tokenization
      text (str):  text to transform in a spaCy doc class.

    Returns: ---
  """
  doc = model(text)
  print (f"The text:\n\n{get_text_to_print(text)}\n\nwas converted in a spaCy object: {type(doc)}\n")
  print (f"Token-based analysis. Each token is a spaCy object: {type(doc[0])}\n")

  # We obtain rows to print: headers and content
  rows  = []
  # head_align: List of tuples. Each tuple: heather and its alignment when printing
  head_align  = [('Token', '<'), ('Lemma', '<'), ('Syntactic parent', '<'), ('#Tok', '>'), ('Chr_Start', '>'), ('Chr_End', '>'), ('POS', '<'),
                 ('TAG', '<'), ('TAG meaning:', '<'), ('ENT', '<'), ('DEP', '<'), ('DEP meaning:', '<')]
  head, align = list(zip(*head_align))
  rows.append(head)                           # Header
  rows.append(['='*len(i) for i in head])     # Underline headers
  for tok in doc:
    rows.append([tok.text, tok.lemma_, tok.head.text, str(tok.i), str(tok.idx), str(tok.idx+len(tok)-1), tok.pos_,
                 tok.tag_, str(spacy.explain(tok.tag_))[:20], tok.ent_type_, tok.dep_, str(spacy.explain(tok.dep_))[:20]])

  # Width of each column: the witdh of the longest element
  columns       = zip(*rows)
  column_widths = [max(len(i) for i in col) for col in columns]

  # Print the files with alignment
  for row in rows:
    print(*[f"{row[i]:{align[i]}{column_widths[i]}}  " for i in range(0, len(row))])

In [ ]:
def get_text_to_print(text):
  """Format given text.

    Parameters:
      text (str): text to print

    Returns:
      str: text formatted in 100 character lines with an initial line numbering the characters
  """
  line_length = 100
  line_poss   = "     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100"
  text        = text.replace("\n", " ")     # In order to avoid that the \n character produces a line change.
  text        = text.replace("\r", " ")     # In wikipedia texts we have detected the character '\r' that, if interpreted, may induce some printing problems.
  text_format = "\n".join([ f"{i//line_length:<5}{text[i:i+line_length]}"  for i in range(0, len(text), line_length) ])
  return line_poss + "\n" + text_format + "\n" + line_poss

**c. Carregar el model `en_core_web_sm`**

**Resultat esperat**: Codi per a la càrrega del model.

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################

import spacy

**d. Convertir un text en objecte `Doc` de spaCy.**

Per a realitzar la detecció d'entitats anomenades, proposar un text en anglès que mencioni a entitats de diferent tipus.

**Sortida Esperada**: Visualitzar els resultats d'analitzar el text proposat a nivell de cada POS.

**Suggeriment:** Per a la visualització es pot utilitzar la funció *get_tokens_to_print()*, prèviament creada, o *displacy.render()* de spaCy.

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################




###2.1.2 Entrenar un nou model de NER amb el corpus escollit de detecció d'entitats.

**a. Convertir el corpus escollit al 'format spaCy'.**

*Suggeriment:* Si el corpus es gran i l'entrenament triga massa, pots generar una versió més reduïda de tots els arxius del corpus escollit (train, dev y test). La reducció es podria fer del 25%.

Recorda que SpaCy conté funcions que permeten convertir formats, com ara *conll*, al format compilat que necessita el mòdul de train de spaCy.

**Sortida esperada:** Codi per a convertir el corpus training (train.txt) i el de validació (valid.txt), del format origen a format spaCy.

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################



**b. Descarregar el model `en_core_web_trf`**.

In [ ]:
!python -m spacy download en_core_web_trf

**Important:**

* La descàrrega de en_core_web_trf pot requerir instal·lar prèviament `spacy-curated-transformers`
* A Google Colab, si després de la descàrrega del model `en_core_web_trf` s'obté el missatge 'Restart to reload dependencies', haurà d'executar-se l'opció de menú 'Runtime>>>Restart session'. A continuació, per a major seguretat, tornar a executar la secció 0 d'aquest notebook.

In [ ]:
!python -m spacy info en_core_web_trf

In [ ]:
!python -m spacy validate

In [ ]:
import spacy
print (f"Spacy version installed: {spacy.__version__}")

**c. Verificar si es pot utilitzar GPU**

Si aquest notebook s'està executant en una GPU, al paràmetre 'gpu-id' se li assignarà 0. Amb aquest canvi, SpaCy utilitzarà la GPU i s'accelerarà el temps d'entrenament.

In [ ]:
import torch
if torch.cuda.is_available():
    gpu_id = 0    # Use the first available GPU
    print(f"Using GPU {gpu_id}")
else:
    gpu_id = -1   # Use CPU
    print("Using CPU")

**d. Entrenar el model `en_core_web_trf` utilitzant la funció train de spaCy.**


SpaCy realitza l'entrenament del model d'acord amb els paràmetres de l'arxiu de configuración 'config.cfg'. Una descripció de les seccions d'aquesta configuració la pots trobar a: https://spacy.io/usage/training#config; descriu algun canvi que realitzaries si creus que pots millorar l'entrenament.

A més, considera que a l'arxiu 'config.cfg' no es determina un nombre prefixat d'*epochs* (veure sección [training] de 'config.cfg'). El criteri per a finalitzar el determinen els paràmetres *max_steps* (20000) i *patience* (1600). Aquests paràmetres determinen que l'entrenament finalitzarà si es compleix una de les condicions:
* quan s'hagin processat 20000 'batches', o bé
* quan, després de 1600 'batches' processats, no hi ha hagut millora en el model.

**`Interrupció del procés`**

Per restriccions de temps de procés (o per no consumir les *compute unit* de les que disposem a Google Colab), si s'observa que transcorregudes diverses iteracions (per exemple a partir de la tercera) el model ha anat millorant (columna SCORE) y té un valor superior a 0.9, interrompre manualment el procés. El 'best model' fins a aquesta iteració s'emmagatzemarà al path '{my_path_pra2}/NER/output_ner/model-best'. Aquest model és el que s'utilitzarà per a validar o predir

In [ ]:
# Training

!python -m spacy train {my_path_pra2}/NER/config.cfg \
    --output {my_path_pra2}/NER/output_ner/ \
    --paths.train {my_path_pra2}/NER/train.spacy  \
    --gpu-id {gpu_id} \
    --paths.dev {my_path_pra2}/NER/valid.spacy

**e. Predicció d'un text d'exemple (inferència amb el model entrenat).**

Carregar el millor model entrenat i utilitzar-lo per a predir una frase d'exemple. Recordar que, si el model ja està entrenat y emmagatzemat en una carpeta local, és millor, a l'iniciar una nova sessió, recuperar la millor versió entrenada des de local.

**Resultat esperat:** visualitzar els resultats de la predicció de la variable `text`.


In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################

text = """\
Next Tuesday, Ana plans to travel to London to start her new job at Google. \
Apple announced earnings of $5 billion from iPhone sales in Cupertino.\
Jules Verne visited the Eiffel Tower while he was in Paris.\
"""




**f. Avaluar els resultats obtinguts i calcular les mètriques.**

**Sortida Esperada**: Càlcul de les mètriques utilitzant les dades de prova.

**Important:** Abans del càlcul de les mètriques, no oblidar convertir el format de l'arxiu test.txt al 'format spaCy'.

In [ ]:
#Usar GPU si és possible:

import torch
if torch.cuda.is_available():
    gpu_id = 0    # Use the first available GPU
    print(f"Using GPU {gpu_id}")
else:
    gpu_id = -1   # Use CPU
    print("Using CPU")

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################



## 2.2 NEL (1 punt)

En aquesta secció, obtindrem els enllaços a Wikidata relacionats amb les entitats que s'obtenen utilitzant spaCy.

Es desenvoluparà una funció que, donat un text, obtingui automàticament les entitats i les relacionarà amb la corresponent entrada a Wikidata.
Per a implementar la solució, podeu usar, por exemple, la llibreria `wikidata.client`, o realitzar sol·licituds directes a la API, utilitzant la llibreria `requests`.

**Important:** En el `Notebook d'Exemple` es proporciona un text en català, és per això que es carrega el model `ca_core_news_sm`. Per a analitzar frases en castellà, canviar el model, per ejemplo, usar `es_core_news_sm`. A més, si s'utilitza `requests`, modificar l'idioma a la URL d'accés al EndPoint de WikiData.

**Sortida Esperada**: Llista de les entitats reconegudes en el text, amb la seva respectiva `URI`.


In [ ]:
nel_model = "ca_core_news_sm" # Model a usar, es pot canvar a "es_core_news_sm" per a processar frases en castellà.

!pip install wikidata
!python -m spacy download '{nel_model}'

In [ ]:
#############################################
# SOLUCIÓ                                   #
#############################################

text = """\
L'escriptor i guionista hongarès László Krasznahorkai \
ha guanyat el Premi Nobel de Literatura 2025.
El recent Nobel, de 71 anys, té la seva primera novel·la "Tango Satànic" (1985) \
traduïda al català per Carles Dachs, editada a Barcelona aquest 2025 per \
Edicions del Cràter, amb seu a Barcelona."""